<a href="https://colab.research.google.com/github/Xage-masa/mistral7b-empathy-lora-project/blob/main/%D0%B4%D0%BE%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8_%D1%81_11%D0%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Подключение к Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
output_dir = "/content/drive/MyDrive/empathy_lora_checkpoints/new_run"


In [ ]:
# Установка нужных библиотек
!pip install transformers datasets peft accelerate bitsandbytes trl


In [ ]:
# Пути к файлам и модель
base_model = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
adapter_path = "/content/drive/MyDrive/empathy_lora_checkpoints/checkpoint-11000"
data_path = "/content/drive/MyDrive/Colab Notebooks/empathy_all.txt"


In [ ]:
from datasets import load_dataset

# empathy_all.txt должен быть в формате "текст<TAB>эмоция"
dataset = load_dataset("csv", data_files={"train": data_path}, delimiter="\t", column_names=["text", "label"])


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token


def preprocess(batch):
    result = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)
    result["labels"] = batch["label"]
    return result

tokenized = dataset["train"].map(preprocess, batched=True)





/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/79548 [00:00<?, ? examples/s]

In [ ]:
print(tokenized[0].keys())


dict_keys(['text', 'label', 'input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import PeftModel

# Загружаем базовую модель и накладываем LoRA
model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=28)  # или нужное количество эмоций
model.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(model, adapter_path)
model.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at NousResearch/Nous-Hermes-2-Mistral-7B-DPO and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 0 || all params: 7,114,190,848 || trainable%: 0.0000


In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,  # <-- теперь чекпоинты сохраняются в Google Drive
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,
    save_steps=1000,
    save_total_limit=3,  # максимум 3 последних чекпоинта, чтобы не засорять диск
    num_train_epochs=1,
    fp16=True,
    resume_from_checkpoint="/content/drive/MyDrive/empathy_lora_checkpoints/checkpoint-11000"

)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

trainer.train()



In [ ]:
output_dir = "/content/drive/MyDrive/finetuned-empathy-companion"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
